<a href="https://colab.research.google.com/github/Innovatewithapple/TransformersProjects/blob/main/TransformerTranslation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input,Embedding,Layer,Dense
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
subjects = ["I", "You", "He", "She", "We", "They"]
verbs = ["eat", "like", "see", "have", "want", "buy", "find", "need"]
objects = ["apple", "car", "book", "coffee", "house", "dog", "cat", "food"]

# English → Spanish mapping (simple)
translation = {
    "I": "Yo", "You": "Tú", "He": "Él", "She": "Ella", "We": "Nosotros", "They": "Ellos",
    "eat": "como", "like": "gusta", "see": "veo", "have": "tengo",
    "want": "quiero", "buy": "compro", "find": "encuentro", "need": "necesito",
    "apple": "manzana", "car": "coche", "book": "libro", "coffee": "café",
    "house": "casa", "dog": "perro", "cat": "gato", "food": "comida"
}

input_texts = []
target_texts = []

for s in subjects:
    for v in verbs:
        for o in objects:
            eng = f"{s} {v} {o}"
            spa = f"{translation[s]} {translation[v]} {translation[o]}"
            input_texts.append(eng)
            target_texts.append(spa)

print(len(input_texts))  # 6 * 8 * 8 = 384

384


In [ ]:
extra_phrases = [
    ("Good morning", "Buenos días"),
    ("Thank you", "Gracias"),
    ("I am happy", "Estoy feliz"),
    ("Where is the car", "Dónde está el coche"),
    ("I need help", "Necesito ayuda"),
    ("See you soon", "Hasta pronto"),
    ("I am learning", "Estoy aprendiendo"),
    ("The sky is blue", "El cielo es azul"),
    ("The water is cold", "El agua está fría"),
    ("It is hot today", "Hace calor hoy")
]

for eng, spa in extra_phrases:
    input_texts.append(eng)
    target_texts.append(spa)

In [ ]:
input_texts = input_texts
target_texts = target_texts

In [ ]:
# Add start and end sequence in target text
target_texts = ['startseq ' + t + ' endseq' for t in target_texts]

In [ ]:
target_texts[0]

'startseq Yo como manzana endseq'

In [ ]:
input_token = Tokenizer()
input_token.fit_on_texts(input_texts)
input_sequences = input_token.texts_to_sequences(input_texts)

output_token = Tokenizer()
output_token.fit_on_texts(target_texts)
output_sequences = output_token.texts_to_sequences(target_texts)

In [ ]:
output_sequences[0]

[1, 3, 11, 12, 2]

In [ ]:
input_vocab_length = len(input_token.word_index) + 1
output_vocab_length = len(output_token.word_index) + 1

In [ ]:
input_vocab_length

41

In [ ]:
max_input_length = max(len(seq) for seq in input_sequences)
max_output_length = max(len(seq) for seq in output_sequences)

In [ ]:
# Here we use pad_sequence because we need every array contains same element, and use post so that 0 can be add as empty elemtns in array.
input_sequences = pad_sequences(input_sequences,maxlen=max_input_length,padding='post')
output_sequences = pad_sequences(output_sequences,maxlen=max_output_length,padding='post')

In [ ]:
input_sequences

array([[ 1, 10, 11,  0],
       [ 1, 10,  7,  0],
       [ 1, 10, 12,  0],
       ...,
       [24, 34, 23, 35],
       [24, 36, 23, 37],
       [38, 23, 39, 40]], dtype=int32)

In [ ]:
# Here the idea is in output we have to find spanish or whatever language we are tyring to translate, we need it as decoder where the input is previous word and output is net word.
decoder_input = output_sequences[:,:-1]
decoder_output = output_sequences[:,1:]

decoder_output = np.expand_dims(decoder_output,-1)

In [ ]:
# Attention class
class Attention(Layer):
  def __init__(self,d_model):
      super().__init__()
      self.WQ = Dense(d_model)
      self.WK = Dense(d_model)
      self.WV = Dense(d_model)

  def call(self,q,k,v):
    Q = self.WQ(q)
    K = self.WK(k)
    V = self.WV(v)

    scores = tf.matmul(Q,K,transpose_b=True)
    dk = tf.cast(tf.shape(K)[-1],tf.float32)
    scores = scores / tf.sqrt(dk)

    weights = tf.nn.softmax(scores)
    output = tf.matmul(weights,V)

    return output

In [ ]:
# self-attention
encoder_input = Input(shape=(max_input_length,))
x = Embedding(input_vocab_length,64)(encoder_input)
encoder_output = Attention(64)(x,x,x)

decoder_inputs = Input(shape=(max_output_length -1,))
y = Embedding(output_vocab_length,64)(decoder_inputs)
decoder_outputs = Attention(64)(y,y,y)

#Cross-attention
final_output = Attention(64)(decoder_outputs,encoder_output,encoder_output)

output = Dense(output_vocab_length,activation='softmax')(final_output)

model = Model([encoder_input,decoder_inputs],output)

model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
model.fit([input_sequences,decoder_input],decoder_output,epochs=200,validation_split=0.2)


Epoch 1/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 61ms/step - accuracy: 0.1752 - loss: 3.7455 - val_accuracy: 0.2101 - val_loss: 3.7008
Epoch 2/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.2000 - loss: 3.4957 - val_accuracy: 0.2101 - val_loss: 3.4693
Epoch 3/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.2000 - loss: 3.0572 - val_accuracy: 0.2152 - val_loss: 3.3608
Epoch 4/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.2000 - loss: 2.8818 - val_accuracy: 0.2000 - val_loss: 3.4069
Epoch 5/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.2000 - loss: 2.7836 - val_accuracy: 0.2152 - val_loss: 3.4715
Epoch 6/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.2000 - loss: 2.7254 - val_accuracy: 0.2025 - val_loss: 3.5421
Epoch 7/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.2000 - loss: 2.6874 - val_accuracy: 0.2025 - val_loss: 3.5828
Epoch 8/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.2000 - loss: 2.6465 - val_accuracy: 0.